In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# S03 — Worked coding answers

Three exercises. Learner NotImplementedError cells are assignments, not broken reference code. Read the prose lesson and predict the checks first.

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
from dataclasses import asdict
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/shape_lab').exists())
sys.path.insert(0, str(ROOT / 'src'))
from shape_lab import sports_v8 as sport
from shape_lab.persistence import finite_bottleneck
spl, profiles = sport.load_sports(ROOT)
frames = np.array([r['frame'] for r in spl['records']])
times = sport.frame_times(frames, spl['sampling_rate'])
ball = sport.to_metres([r['ball'] for r in spl['records']], spl['xyz_unit'])
report_dir = ROOT / 'reports/v8/sports'
report_dir.mkdir(parents=True, exist_ok=True)
def save_report(stage, result):
    # Reports contain ordinary finite values; absent values are recorded explicitly.
    (report_dir / f'{stage}.json').write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Data:', len(frames), 'selected frames from one shot;', len(profiles), 'profiles from', profiles.player_id.nunique(), 'players')


Data: 12 selected frames from one shot; 12 profiles from 8 players


## S03.C1 — Construct a full Euclidean pairwise-distance matrix.

Hint: Broadcasting compares every row to every row.

In [3]:
def pairwise_distance(x):
    x=np.asarray(x,float)
    return np.linalg.norm(x[:,None,:]-x[None,:,:],axis=2)

In [4]:
assert np.allclose(pairwise_distance([[0,0],[3,4]]),[[0,5],[5,0]])
print("S03.C1: check passed")

S03.C1: check passed


## S03.C2 — Fit feature means on declared training rows only.

Hint: Never fit on the third row when it belongs to test.

In [5]:
def training_mean(x,rows):
    return np.asarray(x,float)[np.asarray(rows,int)].mean(axis=0)

In [6]:
assert np.allclose(training_mean([[1,2],[3,4],[99,99]],[0,1]),[2,3])
print("S03.C2: check passed")

S03.C2: check passed


## S03.C3 — Return lifetimes of finite positive intervals from a birth/death array.

Hint: This is an explicit reporting policy, not evidence that essential bars do not exist.

In [7]:
def finite_lifetimes(pairs):
    p=np.asarray(pairs,float).reshape(-1,2)
    valid=np.isfinite(p).all(axis=1)&(p[:,1]>p[:,0])
    return p[valid,1]-p[valid,0]

In [8]:
assert np.allclose(finite_lifetimes([[1,2],[0,np.inf],[1,1]]),[1])
print("S03.C3: check passed")

S03.C3: check passed
